# Abstract

**We will predict whether the cancer is benign or malignant by using different feature selection techniques with different models to analyze their impact on model performance.**

# Table of Contents
1. [Introduction to Feature Selection Methods](#1-introduction)
2. [Data Loading & Preprocessing](#2-data-loading)
3. [Exploratory Data Analysis](#3-eda)
4. [Feature Selection Methods](#4-feature-selection)
   - 4.1 Filter Methods (Correlation + Variance)
   - 4.2 Univariate Selection (SelectKBest)
   - 4.3 Feature Importance (Random Forest)
   - 4.4 Principal Component Analysis (PCA)
5. [Results Comparison](#5-results)

## 1. Introduction to Feature Selection Methods <a id="1-introduction"></a>

Feature selection is a crucial step in the machine learning workflow. It involves identifying and selecting the most relevant features from a dataset while removing redundant, irrelevant, or noisy variables. Effective feature selection can improve model performance, reduce overfitting, decrease training time, and enhance model interpretability.

### Common Feature Selection Methods

### 1. Filter Methods

Filter methods evaluate features independently of any machine learning model.

* **Correlation Analysis:** Identifies and removes highly correlated features to reduce multicollinearity.
* **Variance Threshold:** Removes features with little or no variation, as they often provide limited predictive value.

### 2. Univariate Selection

These methods evaluate each feature individually using statistical tests.

* **SelectKBest:** Selects the top *k* features based on statistical scores such as Chi-Square, ANOVA F-test, or Mutual Information.
* **SelectPercentile:** Selects a specified percentage of the highest-scoring features.

### 3. Feature Importance Methods

Some machine learning algorithms provide feature importance scores directly.

* **Tree-Based Models:** Algorithms such as Random Forest and Gradient Boosting rank features according to their contribution to predictions.
* **L1 Regularization (Lasso):** Encourages sparsity by shrinking less important feature coefficients to zero, effectively performing feature selection.

### 4. Principal Component Analysis (PCA)

PCA is a dimensionality reduction technique that transforms the original features into a smaller set of orthogonal components while preserving most of the dataset's variance.

### 5. Recursive Feature Elimination (RFE)

RFE iteratively trains a model, ranks features based on importance, and removes the least important features until the desired number of features remains.

### 6. Forward and Backward Selection

These are wrapper-based methods that evaluate feature subsets using model performance.

* **Forward Selection:** Starts with no features and adds the most informative feature at each iteration.
* **Backward Elimination:** Starts with all features and removes the least informative feature at each iteration.

### 7. Regularization-Based Methods

* **ElasticNet:** Combines L1 (Lasso) and L2 (Ridge) regularization to select important features while maintaining model stability.

### 8. Correlation-Based Selection

Measures the relationship between each feature and the target variable, selecting features with the strongest predictive relationships.

---

### Conclusion

There is no single best feature selection technique for every dataset. The optimal approach depends on the problem, data characteristics, and machine learning model being used. In practice, it is often beneficial to experiment with multiple feature selection methods and compare their impact using appropriate evaluation metrics and cross-validation.


## 2. Data Loading & Preprocessing <a id="2-data-loading"></a>

In [ ]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Scikit-learn metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Scikit-learn models
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, 
    AdaBoostClassifier, 
    GradientBoostingClassifier
)

# XGBoost
from xgboost import XGBClassifier

# Feature selection
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import PCA

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("All libraries imported successfully!")

### 2.1 Load Dataset

In [ ]:
# Load the Breast Cancer Wisconsin dataset
# Note: Update the path according to your environment
data = pd.read_csv('/kaggle/input/breast-cancer-wisconsin-data/data.csv')

# Display first few rows
data.head()

### 2.2 Data Overview

In [ ]:
# Basic information about the dataset
print("Dataset Shape:", data.shape)
print("\nColumn Information:")
data.info()

**Observations:**
- 569 samples with 33 columns
- `id`: Patient identifier (to be removed)
- `diagnosis`: Target variable (M=Malignant, B=Benign)
- `Unnamed: 32`: Empty column (to be removed)
- 30 numerical features related to cell nucleus characteristics

### 2.3 Data Cleaning

In [ ]:
# Drop unnecessary columns
data = data.drop(['id', 'Unnamed: 32'], axis=1)

# Encode target variable: M (Malignant) = 1, B (Benign) = 0
data['diagnosis'] = data['diagnosis'].map({'M': 1, 'B': 0})

# Separate features and target
X = data.drop('diagnosis', axis=1)
y = data['diagnosis']

print(f"Features shape: {X.shape}")
print(f"Target distribution:{y.value_counts()}")

### 2.4 Data Scaling

In [ ]:
# Standardize features to have mean=0 and std=1
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Display first 5 rows of scaled data
X_scaled.head()

## 3. Exploratory Data Analysis <a id="3-eda"></a>

### 3.1 Statistical Summary

In [ ]:
# Descriptive statistics
data.describe().T.round(3)

### 3.2 Target Distribution

In [ ]:
# Visualize target distribution
fig, ax = plt.subplots(figsize=(8, 5))
counts = data['diagnosis'].value_counts()
bars = ax.bar(['Benign (0)', 'Malignant (1)'], counts.values, color=['skyblue', 'salmon'])
ax.bar_label(bars, labels=counts.values)
ax.set_title('Distribution of Diagnosis', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### 3.3 Helper Functions for Visualization

In [ ]:
def plot_correlation_heatmap(columns, title="Correlation Heatmap"):
    """
    Plot correlation heatmap for given columns.
    """
    features = list(columns)
    plt.figure(figsize=(10, 8))
    sns.heatmap(data[features].corr(), annot=True, fmt='.1f', cmap='coolwarm', center=0)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_pairplot(columns, title="Pairplot"):
    """
    Plot pairplot for given columns.
    """
    g = sns.pairplot(columns, diag_kind='kde', plot_kws={'alpha': 0.6})
    g.fig.suptitle(title, y=1.02, fontsize=14, fontweight='bold')
    plt.show()


def plot_swarmplot(columns, target=y):
    """"
    Plot swarmplot to visualize feature distribution by diagnosis.
    """
    df = pd.concat([target, columns], axis=1)
    df_melted = pd.melt(df, id_vars="diagnosis", var_name="features", value_name='value')
    
    plt.figure(figsize=(12, 8))
    sns.swarmplot(x="features", y="value", hue="diagnosis", data=df_melted, size=3)
    plt.xticks(rotation=90)
    plt.title('Feature Distribution by Diagnosis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Feature Selection Methods <a id="4-feature-selection"></a>

### Classifier Setup

In [ ]:
# Define classifiers with consistent random states
classifiers = {
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(random_state=43, max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVC': SVC(random_state=43),
    'Random Forest': RandomForestClassifier(random_state=41, n_estimators=100),
    'XGBoost': XGBClassifier(random_state=43, use_label_encoder=False, eval_metric='logloss'),
    'AdaBoost': AdaBoostClassifier(random_state=43, n_estimators=100),
    'Neural Network': MLPClassifier(random_state=43, max_iter=1000, hidden_layer_sizes=(100,)),
    'Gradient Boosting': GradientBoostingClassifier(random_state=45, n_estimators=100)
}


def evaluate_classifiers(X_train, X_test, y_train, y_test, method_name=""):
    """
    Train and evaluate all classifiers on given data.
    Returns a DataFrame with results.
    """
    results = []
    
    for name, classifier in classifiers.items():
        # Train model
        classifier.fit(X_train, y_train)
        y_pred = classifier.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        
        results.append({
            'Method': method_name,
            'Classifier': name,
            'Accuracy': accuracy * 100,
            'Precision': precision * 100,
            'Recall': recall * 100,
            'F1 Score': f1 * 100
        })
    
    return pd.DataFrame(results)

---

### 4.1 Method 1: Filter Methods (Correlation + Variance) <a id="filter-methods"></a>

**Approach:**
1. Remove highly correlated features (correlation > 0.9)
2. Remove low-variance features
3. Group features by type: mean, standard error (se), and worst

#### 4.1.1 Mean Features Analysis

In [ ]:
# Mean features (columns 1-10 in original data, excluding diagnosis)
mean_features = data.columns[1:11]  # radius_mean to fractal_dimension_mean
print("Mean features:", list(mean_features))

# Correlation heatmap
plot_correlation_heatmap(mean_features, "Mean Features - Correlation Heatmap")

In [ ]:
# Pairplot for mean features
plot_pairplot(X_scaled.iloc[:, :10], "Mean Features - Pairplot")

In [ ]:
# Swarmplot for mean features
plot_swarmplot(X_scaled.iloc[:, :10])

**Drop decisions for Mean Features:**
- `radius_mean`, `area_mean`: Highly correlated with perimeter
- `texture_mean`, `smoothness_mean`, `symmetry_mean`, `fractal_dimension_mean`: Low variance / less discriminative

#### 4.1.2 Standard Error Features Analysis

In [ ]:
# SE features (columns 11-20)
se_features = data.columns[11:21]
print("SE features:", list(se_features))

plot_correlation_heatmap(se_features, "SE Features - Correlation Heatmap")

In [ ]:
plot_pairplot(X_scaled.iloc[:, 10:20], "SE Features - Pairplot")

In [ ]:
plot_swarmplot(X_scaled.iloc[:, 10:20])

**Drop decisions for SE Features:**
- `radius_se`, `perimeter_se`: Highly correlated
- `concave points_se`, `smoothness_se`, `concavity_se`, `symmetry_se`, `texture_se`: Low variance

#### 4.1.3 Worst Features Analysis

In [ ]:
# Worst features (columns 21-30)
worst_features = data.columns[21:31]
print("Worst features:", list(worst_features))

plot_correlation_heatmap(worst_features, "Worst Features - Correlation Heatmap")

In [ ]:
plot_pairplot(X_scaled.iloc[:, 20:30], "Worst Features - Pairplot")

In [ ]:
plot_swarmplot(X_scaled.iloc[:, 20:30])

**Drop decisions for Worst Features:**
- `perimeter_worst`, `radius_worst`: Highly correlated with area
- `compactness_worst`, `texture_worst`, `smoothness_worst`, `symmetry_worst`, `fractal_dimension_worst`: Low variance

#### 4.1.4 Apply Filter Method & Evaluate

In [ ]:
# Create a copy for filter method
X_filter = X.copy()

# Drop highly correlated and low variance features
columns_to_drop = [
    # Mean features
    'radius_mean', 'area_mean', 'texture_mean', 'smoothness_mean', 
    'symmetry_mean', 'fractal_dimension_mean',
    # SE features
    'radius_se', 'perimeter_se', 'concave points_se', 'smoothness_se',
    'concavity_se', 'symmetry_se', 'texture_se',
    # Worst features
    'perimeter_worst', 'radius_worst', 'compactness_worst', 'texture_worst',
    'smoothness_worst', 'symmetry_worst', 'fractal_dimension_worst'
]

X_filter = X_filter.drop(columns=columns_to_drop, errors='ignore')

print(f"Original features: {X.shape[1]}")
print(f"Remaining features after filtering: {X_filter.shape[1]}")
print(f""
"Remaining features:", list(X_filter.columns))

In [ ]:
# Visualize remaining features
from pandas.plotting import scatter_matrix

color_map = {0: "blue", 1: "red"}
colors = data["diagnosis"].map(lambda x: color_map.get(x))

scatter_matrix(X_filter, c=colors, alpha=0.5, figsize=(15, 15), diagonal='kde')
plt.suptitle('Scatter Matrix of Filtered Features', fontsize=16, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Train-test split and evaluate
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_filter, y, test_size=0.1, random_state=42, stratify=y
)

results_filter = evaluate_classifiers(
    X_train_f, X_test_f, y_train_f, y_test_f, "Filter Method"
)

print("=== Filter Method Results ===")
print(results_filter.to_string(index=False))

---

### 4.2 Method 2: Univariate Selection (SelectKBest with Chi-Squared) <a id="univariate"></a>

SelectKBest selects the top k features based on univariate statistical tests. We use chi-squared test which measures dependence between categorical variables.

In [ ]:
# Train-test split on original features
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# Apply SelectKBest with chi-squared test
# Note: chi2 requires non-negative features, so we use original (unscaled) data
selector = SelectKBest(score_func=chi2, k=5)
selector.fit(X_train_u, y_train_u)

# Get selected feature names
selected_mask = selector.get_support()
selected_features = X_train_u.columns[selected_mask]

print("Selected Top 5 Features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")

In [ ]:
# Transform data to selected features
X_train_selected = X_train_u[selected_features]
X_test_selected = X_test_u[selected_features]

# Evaluate
results_univariate = evaluate_classifiers(
    X_train_selected, X_test_selected, y_train_u, y_test_u, "Univariate Selection"
)

print("=== Univariate Selection Results ===")
print(results_univariate.to_string(index=False))

---

### 4.3 Method 3: Feature Importance (Random Forest) <a id="feature-importance"></a>

Random Forest provides built-in feature importance scores based on how much each feature contributes to reducing impurity across all trees.

In [ ]:
# Train-test split
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# Train Random Forest to get feature importances
rf = RandomForestClassifier(random_state=43, n_estimators=100)
rf.fit(X_train_i, y_train_i)

# Get feature importances
importances = rf.feature_importances_
std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)

# Sort by importance and get top 10
indices = np.argsort(importances)[::-1][:10]
top_features = X_train_i.columns[indices]

print("Top 10 Most Important Features:")
for i, feat in enumerate(top_features, 1):
    print(f"  {i}. {feat} ({importances[indices[i-1]]:.4f})")

In [ ]:
# Plot feature importances
plt.figure(figsize=(12, 8))
plt.bar(range(10), importances[indices], color='forestgreen',yerr=std[indices], align='center', alpha=0.8, ecolor='black')
plt.xticks(range(10), top_features, rotation=45, ha='right')
plt.xlim([-1, 10])
plt.title('Top 10 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
plt.ylabel('Importance Score')
plt.xlabel('Features')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate with top features
X_train_top = X_train_i[top_features]
X_test_top = X_test_i[top_features]

results_importance = evaluate_classifiers(
    X_train_top, X_test_top, y_train_i, y_test_i, "Feature Importance"
)

print("=== Feature Importance Results ===")
print(results_importance.to_string(index=False))

---

### 4.4 Method 4: Principal Component Analysis (PCA) <a id="pca"></a>

PCA transforms features into a lower-dimensional space while retaining maximum variance. We use 2 components for visualization and evaluation.

In [ ]:
# Train-test split
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# Standardize data
scaler_pca = StandardScaler()
X_train_scaled_p = scaler_pca.fit_transform(X_train_p)
X_test_scaled_p = scaler_pca.transform(X_test_p)

# Fit PCA to determine optimal components
pca_full = PCA()
pca_full.fit(X_train_scaled_p)

# Plot explained variance ratio
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1),pca_full.explained_variance_ratio_, 'bo-', linewidth=2, markersize=6)
plt.xlabel('Number of Components', fontsize=12)
plt.ylabel('Explained Variance Ratio', fontsize=12)
plt.title('PCA Explained Variance by Component', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print cumulative variance
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)
print(f"First 2 components explain {cumulative_var[1]*100:.2f}% of variance")
print(f"First 5 components explain {cumulative_var[4]*100:.2f}% of variance")

In [ ]:
# Apply PCA with 2 components
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled_p)
X_test_pca = pca.transform(X_test_scaled_p)

# Visualize PCA projection
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_train_pca[:, 0], X_train_pca[:, 1],c=y_train_p, cmap='coolwarm', alpha=0.7, edgecolors='k', s=50)
plt.colorbar(scatter, label='Diagnosis (0=Benign, 1=Malignant)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
plt.title('PCA: First Two Principal Components', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate classifiers on PCA-transformed data
results_pca = evaluate_classifiers(
    X_train_pca, X_test_pca, y_train_p, y_test_p, "PCA"
)

print("=== PCA Results ===")
print(results_pca.to_string(index=False))

---

## 5. Results Comparison <a id="5-results"></a>

In [ ]:
# Combine all results
all_results = pd.concat([
    results_filter,
    results_univariate,
    results_importance,
    results_pca
], ignore_index=True)

# Display combined results
print("=== Complete Results Summary ===")
print(all_results.to_string(index=False))

In [ ]:
# Create comparison visualization
pivot_accuracy = all_results.pivot(
    index='Classifier', columns='Method', values='F1 Score'
)

# Reorder columns for consistent display
method_order = ['Filter Method', 'Univariate Selection', 'Feature Importance', 'PCA']
pivot_accuracy = pivot_accuracy[method_order]

# Plot grouped bar chart
fig, ax = plt.subplots(figsize=(14, 8))

bar_width = 0.2
index = np.arange(len(pivot_accuracy.index))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for i, (method, color) in enumerate(zip(method_order, colors)):
    ax.bar(index + i * bar_width, pivot_accuracy[method],bar_width, label=method, color=color, alpha=0.85, edgecolor='black')

ax.set_xlabel('Classifiers', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=14, fontweight='bold')
ax.set_title('Comparison of Feature Selection Methods Across Classifiers',fontsize=16, fontweight='bold')
ax.set_xticks(index + bar_width * 1.5)
ax.set_xticklabels(pivot_accuracy.index, rotation=45, ha='right')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=11)
ax.set_ylim([85, 102])
ax.grid(True, axis='y', alpha=0.3)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', padding=3, fontsize=8, rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
# Find best performing combinations
best_by_method = all_results.loc[all_results.groupby('Method')['F1 Score'].idxmax()]
best_overall = all_results.loc[all_results['F1 Score'].idxmax()]

print("=== Best Classifier per Method ===")
print(best_by_method[['Method', 'Classifier', 'Accuracy', 'F1 Score']].to_string(index=False))

print("\n=== Best Overall Combination ===")
print(f"Method: {best_overall['Method']}")
print(f"Classifier: {best_overall['Classifier']}")
print(f"Accuracy: {best_overall['Accuracy']:.2f}%")
print(f"Precision: {best_overall['Precision']:.2f}%")
print(f"Recall: {best_overall['Recall']:.2f}%")
print(f"F1 Score: {best_overall['F1 Score']:.2f}%")